#### Set Up

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import torch
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple
import os, pathlib, shutil, random
import string

In [3]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


Language Translator - English to Spanish

#### Load Dataset

In [4]:
zip_path = keras.utils.get_file(origin=("http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"), fname="spa-eng", extract=True, )
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"
text_path

WindowsPath('C:/Users/PRASHANTH N/.keras/datasets/spa-eng/spa-eng/spa.txt')

In [5]:
with open(text_path, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))
text_pairs[22020]

('Tom lit a cigarette.', '[start] Tomás encendió un cigarrillo. [end]')

#### Data processing

In [6]:
random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples

train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [7]:
# Learning token vocabularies for English and Spanish text
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )
vocab_size = 15000
sequence_length = 20

In [8]:
english_tokenizer = keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

spanish_tokenizer = keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

In [9]:
# Tokenizing and preparing the translation data
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [10]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)
print(inputs["spanish"].shape)

(64, 20)
(64, 20)


####  Attention Mechanism

In [11]:
# Building a sequence-to-sequence encoder
embed_dim = 256
hidden_dim = 1024
source = keras.Input(shape=(None,), dtype="int32", name="english")
source_x = keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)

In [12]:
# Building a sequence-to-sequence decoder
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
target_x = keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)


In [13]:
def dot_product_attention(target, source):
    scores = np.einsum("btd,bsd->bts", target, source)
    scores = keras.ops.softmax(scores, axis=-1)
    return np.einsum("bts,bsd->btd", scores, source)

# dot_product_attention(target, source)

In [14]:
dim = embed_dim
query_dense = keras.layers.Dense(dim)
key_dense = keras.layers.Dense(dim)
value_dense = keras.layers.Dense(dim)
output_dense = keras.layers.Dense(dim)

def parameterized_attention(query, key, value):
    query = query_dense(query)
    key = key_dense(key)
    value = value_dense(value)
    scores = np.einsum("btd,bsd->bts", query, key)
    scores = keras.ops.softmax(scores, axis=-1)
    outputs = np.einsum("bts,bsd->btd", scores, value)
    return output_dense(outputs)

# parameterized_attention(query=target, key=source, value=source)

In [15]:
head_dim = embed_dim
num_heads = 2

query_dense = [keras.layers.Dense(head_dim) for i in range(num_heads)]
key_dense = [keras.layers.Dense(head_dim) for i in range(num_heads)]
value_dense = [keras.layers.Dense(head_dim) for i in range(num_heads)]
output_dense = keras.layers.Dense(head_dim * num_heads)

def multi_head_attention(query, key, value):
    head_outputs = []
    for i in range(num_heads):
        query = query_dense[i](query)
        key = key_dense[i](key)
        value = value_dense[i](value)
        scores = np.einsum("btd,bsd->bts", target, source)
        scores = keras.ops.softmax(scores / math.sqrt(head_dim), axis=-1)
        head_output = np.einsum("bts,bsd->btd", scores, source)
        head_outputs.append(head_output)
    outputs = keras.ops.concatenate(head_outputs, axis=-1)
    return output_dense(outputs)

# multi_head_attention(query=target, key=source, value=source)

In [16]:
# Same as above from keras builtin
"""
multi_head_attention = keras.layers.MultiHeadAttention(num_heads=num_heads, head_dim=head_dim,)
multi_head_attention(query=target, key=source, value=source)
"""

'\nmulti_head_attention = keras.layers.MultiHeadAttention(num_heads=num_heads, head_dim=head_dim,)\nmulti_head_attention(query=target, key=source, value=source)\n'

In [17]:
# Self-Attention
# multi_head_attention(key=source, value=source, query=source)

#### Model - Encoder Block

In [18]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = keras.layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = keras.layers.LayerNormalization()
        self.feed_forward_1 = keras.layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = keras.layers.Dense(hidden_dim)
        self.feed_forward_layernorm = keras.layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Model - Decoder Block

In [19]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = keras.layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = keras.layers.LayerNormalization()
        self.cross_attention = keras.layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = keras.layers.LayerNormalization()
        self.feed_forward_1 = keras.layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = keras.layers.Dense(hidden_dim)
        self.feed_forward_layernorm = keras.layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(query=x, key=source, value=source, attention_mask=mask)
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Model - Complete

In [20]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = keras.layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(source=x, source_mask=source != 0, )

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = keras.layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(target=x, source=encoder_output, source_mask=source != 0, )

x = keras.layers.Dropout(0.5)(x)
target_predictions = keras.layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)
transformer.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  3,840,000 │ english[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spanish             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 256) │  1,315,072 │ embedding_2[0][0… │
│ (TransformerEncode… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  3,840,000 │ spanish[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decoder │ (None, None, 256) │  1,578,752 │ transformer_enco… │
│ (TransformerDecode… │                   │            │ not_equal_3[0][0… │
│                     │                   │            │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, None, 256) │          0 │ transformer_deco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, None,      │  3,855,000 │ dropout_3[0][0]   │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,428,824 (55.04 MB)

 Trainable params: 14,428,824 (55.04 MB)

 Non-trainable params: 0 (0.00 B)

#### Training

In [21]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 118s 90ms/step - accuracy: 0.3535 - loss: 1.5071 - val_accuracy: 0.4933 - val_loss: 1.0392
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 118s 91ms/step - accuracy: 0.5316 - loss: 0.9744 - val_accuracy: 0.5716 - val_loss: 0.8277
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 116s 89ms/step - accuracy: 0.6027 - loss: 0.7626 - val_accuracy: 0.6022 - val_loss: 0.7492
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 116s 89ms/step - accuracy: 0.6474 - loss: 0.6351 - val_accuracy: 0.6108 - val_loss: 0.7335
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 121s 93ms/step - accuracy: 0.6803 - loss: 0.5483 - val_accuracy: 0.6199 - val_loss: 0.7235
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 158s 121ms/step - accuracy: 0.7072 - loss: 0.4843 - val_accuracy: 0.6251 - val_loss: 0.7149
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 118s 91ms/step - accuracy: 0.7295 - loss: 0.4335 - val_accuracy: 0.6290 - val_loss: 0.7333
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 128s 98ms/step - accuracy:

#### Positional Embedding

In [22]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = keras.layers.Embedding(input_dim, output_dim)
        self.position_embeddings = keras.layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = keras.ops.cumsum(keras.ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

#### Model with Positional Embedding

In [23]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(source=x, source_mask=source != 0, )
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(target=x, source=encoder_output,
                                                                source_mask=source != 0, )
x = keras.layers.Dropout(0.5)(x)
target_predictions = keras.layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)
transformer.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ english[0][0]     │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spanish             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │  1,319,176 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ spanish[0][0]     │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 256) │  1,582,856 │ transformer_enco… │
│ (TransformerDecode… │                   │            │ not_equal_5[0][0… │
│                     │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, None, 256) │          0 │ transformer_deco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, None,      │  3,855,000 │ dropout_7[0][0]   │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,447,272 (55.11 MB)

 Trainable params: 14,447,272 (55.11 MB)

 Non-trainable params: 0 (0.00 B)

#### Train New model

In [24]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 125s 96ms/step - accuracy: 0.3847 - loss: 1.4331 - val_accuracy: 0.5315 - val_loss: 0.9622
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.5693 - loss: 0.9007 - val_accuracy: 0.6148 - val_loss: 0.7537
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 103s 79ms/step - accuracy: 0.6406 - loss: 0.7008 - val_accuracy: 0.6486 - val_loss: 0.6748
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 102s 78ms/step - accuracy: 0.6812 - loss: 0.5855 - val_accuracy: 0.6607 - val_loss: 0.6414
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 103s 79ms/step - accuracy: 0.7094 - loss: 0.5076 - val_accuracy: 0.6695 - val_loss: 0.6274
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 103s 79ms/step - accuracy: 0.7328 - loss: 0.4503 - val_accuracy: 0.6769 - val_loss: 0.6188
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 99s 76ms/step - accuracy: 0.7514 - loss: 0.4051 - val_accuracy: 0.6805 - val_loss: 0.6247
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 107s 82ms/step - accuracy: 0

#### Inference

In [25]:
spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
The furniture in his office is very modern.
[start] los muebles en su oficina es muy moderna [end]
-
I'm looking forward to next season.
[start] espero con ganas la próxima estación [end]
-
Big men are not necessarily strong men.
[start] los hombres grandes no son necesariamente muy fuertes [end]
-
Tom wanted to know if Mary had a boyfriend.
[start] tom quería saber si mary le hubiera tenido un novio [end]
-
Anyway, you'll never know.
[start] de cualquier caso pides te lo que quieras [end]


In [27]:
prompt = "Did you eat?"
generate_translation(prompt)

'[start] comiste [end]'